## 1. Datu basetik irudirik gabeko erabiltzaileak lortu

Hurrengo gelaxkak `bdweb` datu basera konektatzen da `mysql-connector-python` liburutegia erabiliz
([w3schools: Python MySQL Select](https://www.w3schools.com/python/python_mysql_select.asp)).

- `users` taulan `SELECT` bat egiten du, profilean irudirik **ez** duten erabiltzaileak bilatzeko.
- Irudirik gabekotzat hartzen dira: `irudia` eremua `NULL` edo hutsik dagoenean, edo irudi lehenetsia (`usuario.jpg`) duenean, hau da, erabiltzaileak ez duenean berea igo.
- `fetchall()`-ekin emaitza guztiak `erabiltzaileak` zerrendan gordetzen dira (hiztegi bat erabiltzaile bakoitzeko), hurrengo urratsean maila bidaltzeko.
- Azkenik konexioa ixten da eta aurkitutako erabiltzaileak pantailan erakusten dira.

In [ ]:
import mysql.connector

# Datu basearen konexio datuak (docker-compose.yml-ko bdweb kontenedorea)
DB_CONFIG = {
    "host": "127.0.0.1",
    "port": 3307,
    "user": "root",
    "password": "",
    "database": "bdweb",
}

# Erabiltzaileak irudia igo ez duenean daukan irudi lehenetsia
IRUDI_LEHENETSIA = "usuario.jpg"

konexioa = mysql.connector.connect(**DB_CONFIG)
kurtsorea = konexioa.cursor(dictionary=True)
kurtsorea.execute(
    "SELECT id, username, izena, abizena, irudia FROM users "
    "WHERE irudia IS NULL OR TRIM(irudia) = '' OR irudia = %s",
    (IRUDI_LEHENETSIA,),
)
erabiltzaileak = kurtsorea.fetchall()
kurtsorea.close()
konexioa.close()

print(f"Irudirik gabeko erabiltzaileak: {len(erabiltzaileak)}")
for e in erabiltzaileak:
    print(f" - {e['id']}: {e['izena']} {e['abizena']} <{e['username']}> (irudia: {e['irudia']!r})")

## 2. Erabiltzaile bakoitzari maila bidali

Hurrengo gelaxkak aurreko urratsean lortutako erabiltzaile bakoitzari mezu informatibo bat bidaltzen dio
([Python docs: email examples](https://docs.python.org/3/library/email.examples.html)).

- `smtplib.SMTP`-rekin Docker-en martxan dagoen **smtp4dev** zerbitzarira konektatzen da (`localhost:2525`).
- Erabiltzaile bakoitzeko `EmailMessage` bat sortzen da: gaia, bidaltzailea, hartzailea (`username` eremua, posta helbidea dena) eta testua.
- Mezuak esaten dio ez duela irudirik igo bere profilera, irudia derrigorrezkoa dela, eta ahal duenean login-a egin eta igotzeko.
- `send_message()`-ekin bidaltzen da. Bidalitako mailak smtp4dev-en web interfazean ikus daitezke: http://localhost:5000

In [ ]:
import smtplib
from email.message import EmailMessage

# smtp4dev zerbitzariaren datuak
SMTP_HOST = "localhost"
SMTP_PORT = 2525
BIDALTZAILEA = "admin@bdweb"

with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as smtp:
    for e in erabiltzaileak:
        mezua = EmailMessage()
        mezua["Subject"] = "Profileko irudia igo behar duzu"
        mezua["From"] = BIDALTZAILEA
        mezua["To"] = e["username"]
        mezua.set_content(
            f"Kaixo {e['izena']},\n\n"
            "Zure profilean ez duzu irudirik igo, eta irudia derrigorrezkoa da.\n"
            "Ahal duzunean, mesedez, egin login-a webgunean eta igo zure profileko irudia.\n\n"
            "Eskerrik asko,\n"
            "bdweb-eko administrazioa\n"
        )
        smtp.send_message(mezua)
        print(f"Maila bidalita: {e['username']}")

print(f"Guztira {len(erabiltzaileak)} maila bidali dira.")